# FP/FiQA Feature Engineering – Canonical Sentiment Dataset

**Purpose:** Single place to load **Financial PhraseBank** and **FiQA**, apply shared preprocessing and feature engineering, and produce a **canonical train/val/test** split. Notebooks **04a** (FinBERT) and **04b** (QNLP) load from this output for apples-to-apples comparison.

**Flow:** Setup → Load raw data (PhraseBank + FiQA) → Normalize text & labels → Optional shared features → Train/val/test split (80/10/10, stratified) → Save to `data/credit_risk_sentiment/processed/`.

**Output:** `train.parquet`, `val.parquet`, `test.parquet` (columns: `text`, `label`, optional `text_len`, `word_count`), and `meta.json` (split sizes, seed).

## 1. Setup: Colab vs local

In [1]:
import os
import sys
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/leemingloon/ocr-agentic-rag.git"
    REPO_DIR = "ocr-agentic-rag"
    if not os.path.isdir(REPO_DIR):
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
    get_ipython().run_line_magic("cd", REPO_DIR)
    get_ipython().system("pip install -q pandas pyarrow")
    ROOT = Path(".").resolve()
    print("Colab: repo ready.")
else:
    ROOT = Path(".").resolve() if (Path(".").resolve() / "credit_risk").exists() else Path("..").resolve()
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    print("Local run. Ensure repo root.")

Local run. Ensure repo root.


## 2. Load raw data (Financial PhraseBank + FiQA)

Same sources and column logic as used previously in 04a/04b: PhraseBank from `FinancialPhraseBank/test`, FiQA from `FiQA/valid`. Concatenate and keep `text`, `label`.

In [2]:
import pandas as pd
import numpy as np

def load_phrasebank(split="test"):
    path = ROOT / "data" / "credit_risk_sentiment" / "FinancialPhraseBank" / split
    if not path.exists():
        return pd.DataFrame()
    pq_files = list(path.glob("*.parquet"))
    if not pq_files:
        return pd.DataFrame()
    df = pd.read_parquet(pq_files[0])
    df["text"] = df.get("text", df.get("sentence", "")).astype(str)
    df["label"] = df.get("label_text", df.get("label", "neutral")).astype(str).str.strip().str.lower()
    return df[["text", "label"]].dropna()

def load_fiqa(split="valid"):
    path = ROOT / "data" / "credit_risk_sentiment" / "FiQA" / split
    if not path.exists():
        return pd.DataFrame()
    pq_files = list(path.glob("*.parquet"))
    if not pq_files:
        return pd.DataFrame()
    df = pd.read_parquet(pq_files[0])
    df["text"] = df.get("sentence", df.get("text", "")).astype(str)
    score = pd.to_numeric(df.get("score", 0), errors="coerce").fillna(0)
    df["label"] = np.where(score < -0.2, "negative", np.where(score > 0.2, "positive", "neutral"))
    return df[["text", "label"]].dropna()

df_pb = load_phrasebank()
df_fiqa = load_fiqa()
df_pb["source"] = "FinancialPhraseBank"
df_fiqa["source"] = "FiQA"
df = pd.concat([df_pb, df_fiqa], ignore_index=True) if len(df_pb) > 0 and len(df_fiqa) > 0 else (df_pb if len(df_pb) > 0 else df_fiqa)
if len(df) == 0:
    raise FileNotFoundError("Download Financial PhraseBank and/or FiQA under data/credit_risk_sentiment/ (see scripts/download_datasets.py)")
print("Financial PhraseBank:", len(df_pb), "| FiQA:", len(df_fiqa), "| Total:", len(df))

Financial PhraseBank: 1000 | FiQA: 117 | Total: 1117


## 3. Shared feature engineering

- **Text:** strip whitespace; keep as-is for FinBERT (case-sensitive).
- **Label:** ensure values in `{"negative", "neutral", "positive"}` (map unknowns to `neutral`).
- **Optional features:** `text_len` (char count), `word_count` for downstream use in 04a/04b if needed.

In [3]:
LABELS_CANONICAL = {"negative", "neutral", "positive"}

df["text"] = df["text"].str.strip()
df["label"] = df["label"].str.strip().str.lower().replace("", "neutral")
df.loc[~df["label"].isin(LABELS_CANONICAL), "label"] = "neutral"

df["text_len"] = df["text"].str.len()
df["word_count"] = df["text"].str.split().str.len().fillna(0).astype(int)

print("Label counts:", df["label"].value_counts().to_dict())
print("Sample:", df[["text", "label", "word_count"]].head(2).to_dict())

Label counts: {'neutral': 628, 'positive': 310, 'negative': 179}
Sample: {'text': {0: 'Rautaruukki said construction group YIT has awarded it a 2.5 mln eur contract to supply the steel structures for a new bridge spanning the Kemijoki river in Northern Finland .', 1: 'Finnish Raute Precision that supplies weighing and dosing systems and plants is changing its name to Lahti Precision .'}, 'label': {0: 'positive', 1: 'neutral'}, 'word_count': {0: 30, 1: 19}}


## 3b. Failure-mode feature columns (for evaluation subsets)

Add columns used by 04a/04b to compute F1 on negation, conditional, numeric comparison, and hedged subsets: `has_negation`, `is_conditional`, `has_numeric_comparison`, `has_hedge`. Optional: requires `credit_risk.sentiment` and optionally `spacy` (pip install spacy && python -m spacy download en_core_web_sm).

In [4]:
# Failure-mode flags for evaluation subsets (04a/04b report F1 on these)
try:
    from credit_risk.sentiment.negation import NegationHandler
    from credit_risk.sentiment.conditional import ConditionalDetector
    from credit_risk.sentiment.numeric_comparison import NumericComparisonExtractor
    from credit_risk.sentiment.hedging import HedgeScorer
    _neg = NegationHandler()
    _cond = ConditionalDetector()
    _num = NumericComparisonExtractor(underperform_pct=5.0)
    _hedge = HedgeScorer()
    df["has_negation"] = df["text"].astype(str).apply(_neg.has_negation)
    df["is_conditional"] = df["text"].astype(str).apply(_cond.is_conditional)
    df["has_numeric_comparison"] = df["text"].astype(str).apply(lambda t: _num.extract(t) is not None)
    df["has_hedge"] = df["text"].astype(str).apply(_hedge.has_hedge)
    print("Failure-mode counts:", df["has_negation"].sum(), "negation,", df["is_conditional"].sum(), "conditional,", df["has_numeric_comparison"].sum(), "numeric,", df["has_hedge"].sum(), "hedged")
except Exception as e:
    print("Failure-mode columns skipped (install credit_risk.sentiment / spacy):", e)
    df["has_negation"] = False
    df["is_conditional"] = False
    df["has_numeric_comparison"] = False
    df["has_hedge"] = False

Failure-mode counts: 35 negation, 52 conditional, 0 numeric, 9 hedged


## 3c. Optional: negation and conditional augmentation for training

For FinBERT fine-tuning (04a), I can add negation-augmented and conditional (neutral) examples to the training set. **Run Section 4 (split) first**, then run **Section 5b** below to build `train_augmented.parquet`; 04a can load it when using augmented fine-tuning.

In [5]:
# No code here; augmentation runs in Section 5b (after split so train_df exists).
pass

## 4. Train/val/test split (80/10/10, stratified)

Single canonical split: `random_state=42`, stratified by `label`. 04a and 04b load these files so splits are identical.

In [6]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
train_df, rest_df = train_test_split(df, test_size=0.2, random_state=RANDOM_STATE, stratify=df["label"])
val_df, test_df = train_test_split(rest_df, test_size=0.5, random_state=RANDOM_STATE, stratify=rest_df["label"])

print("Train:", len(train_df), "| Val:", len(val_df), "| Test:", len(test_df))

Train: 893 | Val: 112 | Test: 112


## 4b. Optional: save augmented training set (for 04a fine-tuning)

Run after Section 4. Creates `train_augmented.parquet` (negation + conditional neutral examples) for use in 04a when fine-tuning with augmentation.

In [7]:
USE_AUGMENTATION = False  # set True to save train_augmented.parquet for 04a
if USE_AUGMENTATION and "train_df" in dir():
    try:
        from credit_risk.sentiment.data_augmentation import add_negation_augmented, add_conditional_neutral_examples
        train_df_aug = add_negation_augmented(train_df.copy(), max_extra=300)
        train_df_aug = add_conditional_neutral_examples(train_df_aug, max_extra=150)
        out_dir = ROOT / "data" / "credit_risk_sentiment" / "processed"
        train_df_aug.to_parquet(out_dir / "train_augmented.parquet", index=False)
        print("Saved train_augmented.parquet, n =", len(train_df_aug))
    except Exception as e:
        print("Augmentation skipped:", e)
else:
    if not USE_AUGMENTATION:
        print("USE_AUGMENTATION=False. Set True and re-run 5b to save train_augmented.parquet.")
    else:
        print("Run Section 4 (split) first so train_df exists.")

USE_AUGMENTATION=False. Set True and re-run 5b to save train_augmented.parquet.


## 5. Save canonical splits

In [8]:
import json

out_dir = ROOT / "data" / "credit_risk_sentiment" / "processed"
out_dir.mkdir(parents=True, exist_ok=True)

train_df.to_parquet(out_dir / "train.parquet", index=False)
val_df.to_parquet(out_dir / "val.parquet", index=False)
test_df.to_parquet(out_dir / "test.parquet", index=False)

meta = {
    "n_train": len(train_df),
    "n_val": len(val_df),
    "n_test": len(test_df),
    "random_state": RANDOM_STATE,
    "version": "1.0",
    "n_total": len(train_df) + len(val_df) + len(test_df),
}
if "source" in train_df.columns:
    for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
        meta[f"n_{split_name}_FinancialPhraseBank"] = int((split_df["source"] == "FinancialPhraseBank").sum())
        meta[f"n_{split_name}_FiQA"] = int((split_df["source"] == "FiQA").sum())
with open(out_dir / "meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Saved to", out_dir)
print("Files:", list(out_dir.glob("*")))

Saved to C:\Users\leemi\OneDrive\Desktop\Job_search_2026\ocr-agentic-rag\data\credit_risk_sentiment\processed
Files: [WindowsPath('C:/Users/leemi/OneDrive/Desktop/Job_search_2026/ocr-agentic-rag/data/credit_risk_sentiment/processed/meta.json'), WindowsPath('C:/Users/leemi/OneDrive/Desktop/Job_search_2026/ocr-agentic-rag/data/credit_risk_sentiment/processed/test.parquet'), WindowsPath('C:/Users/leemi/OneDrive/Desktop/Job_search_2026/ocr-agentic-rag/data/credit_risk_sentiment/processed/train.parquet'), WindowsPath('C:/Users/leemi/OneDrive/Desktop/Job_search_2026/ocr-agentic-rag/data/credit_risk_sentiment/processed/val.parquet')]
